# CaPy v2 Demo — Tri-Modal Contrastive Embeddings

This notebook demonstrates CaPy v2's tri-modal contrastive learning framework.
It loads a pre-trained checkpoint, generates embeddings on synthetic data,
computes retrieval metrics, and visualizes the embedding space.

**Requirements:** CPU only. No GPU or real data needed.

**Checkpoint:** Downloads the T1 seed42 checkpoint from GitHub Releases (~40 MB).
If the download fails, manually place `best.pt` in `checkpoints/demo/`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Download Checkpoint

Downloads the pre-trained T1 (tri-modal) checkpoint from GitHub Releases.
Skip this cell if you already have a checkpoint.

In [ ]:
import urllib.request

CHECKPOINT_DIR = project_root / "checkpoints" / "demo"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "best.pt"

RELEASE_URL = (
    "https://github.com/hoangngo610/CaPy-v2/releases/download/v0.1.0/best.pt"
)

if CHECKPOINT_PATH.exists():
    print(f"Checkpoint already exists: {CHECKPOINT_PATH}")
else:
    print(f"Downloading checkpoint from {RELEASE_URL}...")
    try:
        urllib.request.urlretrieve(RELEASE_URL, CHECKPOINT_PATH)
        size_mb = CHECKPOINT_PATH.stat().st_size / 1e6
        print(f"Downloaded: {CHECKPOINT_PATH} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Falling back to synthetic model (random weights).")
        CHECKPOINT_PATH = None

## 2. Load Model

In [ ]:
from omegaconf import OmegaConf
from src.models.capy import CaPyModel

if CHECKPOINT_PATH and CHECKPOINT_PATH.exists():
    from src.evaluation.report import load_model_and_config
    model, config = load_model_and_config(str(CHECKPOINT_PATH), device="cpu")
    print(f"Loaded trained model (modalities: {model.modalities})")
else:
    # Fallback: create model with default tri-modal config (random weights)
    config = OmegaConf.create({
        "model": {
            "name": "demo_tri_modal",
            "modalities": ["mol", "morph", "expr"],
            "mol_encoder": {"input_dim": 2048, "hidden_dims": [512, 256], "output_dim": 256, "dropout": 0.3},
            "morph_encoder": {"input_dim": 1774, "hidden_dims": [512, 256], "output_dim": 256, "dropout": 0.3},
            "expr_encoder": {"input_dim": 978, "hidden_dims": [512, 256], "output_dim": 256, "dropout": 0.3},
            "projection": {"input_dim": 256, "hidden_dim": 256, "output_dim": 256},
            "embedding_dim": 256,
        },
        "seed": 42,
    })
    model = CaPyModel(config)
    model.eval()
    print("Using random-weight model (checkpoint not available)")

# Get input dimensions from model config
mol_dim = config.model.mol_encoder.input_dim
morph_dim = config.model.morph_encoder.input_dim
expr_dim = config.model.expr_encoder.input_dim
print(f"Input dims: mol={mol_dim}, morph={morph_dim}, expr={expr_dim}")

## 3. Create Synthetic Dataset

We generate synthetic data with the correct shapes to demonstrate the model's
embedding and retrieval pipeline. With a trained checkpoint, the model produces
structured embeddings even on synthetic inputs.

In [ ]:
from src.utils.seeding import seed_everything

seed_everything(42)

N_SAMPLES = 100
N_COMPOUNDS = 20

# Synthetic tensors with correct shapes
mol_data = torch.zeros(N_SAMPLES, mol_dim)
# Simulate ECFP: sparse binary fingerprints (~10% bits on)
for i in range(N_SAMPLES):
    n_on = int(mol_dim * 0.1)
    on_bits = torch.randperm(mol_dim)[:n_on]
    mol_data[i, on_bits] = 1.0

morph_data = torch.randn(N_SAMPLES, morph_dim)
expr_data = torch.randn(N_SAMPLES, expr_dim)

# Compound IDs: 5 samples per compound (simulating dose replicates)
compound_ids = [f"BRD-K{i // 5:08d}" for i in range(N_SAMPLES)]
moa_labels = [f"moa_{i // 5 % 4}" if i < 80 else None for i in range(N_SAMPLES)]

print(f"Synthetic dataset: {N_SAMPLES} samples, {N_COMPOUNDS} compounds")
print(f"Shapes: mol={mol_data.shape}, morph={morph_data.shape}, expr={expr_data.shape}")

## 4. Generate Embeddings

In [ ]:
model.eval()
with torch.no_grad():
    batch = {"mol": mol_data, "morph": morph_data, "expr": expr_data}
    embeddings, encoder_outputs = model(batch)

print("Embedding shapes:")
for mod, emb in embeddings.items():
    norm = emb.norm(dim=-1).mean().item()
    print(f"  {mod}: {emb.shape}, mean L2 norm: {norm:.4f}")

## 5. Compute Retrieval Metrics

In [ ]:
from src.evaluation.report import compute_all_metrics

metrics = compute_all_metrics(embeddings, compound_ids, moa_labels)

# Print key metrics
print("=" * 50)
print("RETRIEVAL METRICS (compound-level R@10)")
print("=" * 50)
for key in sorted(metrics.keys()):
    if key.startswith("compound/") and key.endswith("/R@10") and "->" in key:
        direction = key.split("/")[1]
        print(f"  {direction:20s}  {metrics[key]:.4f}")
if "compound/mean_R@10" in metrics:
    print(f"  {'mean':20s}  {metrics['compound/mean_R@10']:.4f}")

print("\n--- Alignment & Uniformity ---")
for key in sorted(metrics.keys()):
    if key.startswith("align_") or key.startswith("uniform_"):
        print(f"  {key:25s}  {metrics[key]:.4f}")

if "moa/AMI" in metrics:
    print("\n--- MOA Clustering ---")
    for key in sorted(k for k in metrics if k.startswith("moa/")):
        print(f"  {key:25s}  {metrics[key]:.4f}")

## 6. UMAP Visualization

In [ ]:
import umap

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Color by MOA
unique_moas = sorted(set(m for m in moa_labels if m is not None))
cmap = plt.get_cmap("tab10")
moa_to_color = {moa: cmap(i) for i, moa in enumerate(unique_moas)}
grey = (0.7, 0.7, 0.7, 0.5)
colors = [moa_to_color.get(m, grey) if m else grey for m in moa_labels]

for ax, (mod, emb) in zip(axes, sorted(embeddings.items())):
    reducer = umap.UMAP(n_components=2, random_state=42)
    coords = reducer.fit_transform(emb.numpy())
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=8, alpha=0.7)
    ax.set_title(f"UMAP - {mod}")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")

fig.suptitle("Embedding Space by Modality (colored by MOA)", fontsize=14)
fig.tight_layout()
plt.show()

## 7. Cross-Modal Similarity Heatmap

In [ ]:
import itertools

modalities = sorted(embeddings.keys())
pairs = list(itertools.combinations(modalities, 2))

fig, axes = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5))
if len(pairs) == 1:
    axes = [axes]

MAX_SHOW = 50
for ax, (m_a, m_b) in zip(axes, pairs):
    z_a = embeddings[m_a][:MAX_SHOW]
    z_b = embeddings[m_b][:MAX_SHOW]
    sim = (z_a @ z_b.T).numpy()
    im = ax.imshow(sim, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_title(f"{m_a} vs {m_b}")
    ax.set_xlabel(m_b)
    ax.set_ylabel(m_a)
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle("Cross-Modal Cosine Similarity (first 50 samples)", fontsize=14)
fig.tight_layout()
plt.show()

## Summary

This demo showed the CaPy v2 pipeline:
1. Load a pre-trained tri-modal model
2. Generate L2-normalized 256-dim embeddings for all three modalities
3. Compute cross-modal retrieval metrics (R@K, MRR) and MOA clustering
4. Visualize embeddings with UMAP and cross-modal similarity heatmaps

For full training and evaluation on real LINCS data, see:
- `notebooks/colab_training.ipynb` (GPU training on Colab)
- `capy_v2_prd.md` (product requirements)
- `capy_v2_fsd.md` (functional specification)
- `TECHNICAL_REPORT.md` (research write-up)